In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple

@dataclass
class SimConfig:
    """
    Simulation Configuration Table (数据生成配置表)
    """
    n_samples: int = 300           # n: 样本量
    p_features: int = 1000         # p: 特征维度
    rho: float = 0.5               # features pairwise correlation: 特征间的两两相关系数
    num_nonzero: int = 20          # (补充参数) 真正非零系数的数量
    snr: float = 1.0               # target SNR: 目标信噪比 (Medium ≈ 1.0)
    random_seed: int = 42          # 随机种子，保证实验可复现


def generate_simulation_data(config: SimConfig) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate synthetic data based on the provided configuration.
    
    Returns:
        X: Feature matrix of shape (n, p)
        y: Response vector of shape (n,)
        beta_true: The ground truth coefficients of shape (p,)
    """
    # 1. 初始化随机数生成器 (现代 NumPy 最佳实践)
    rng = np.random.default_rng(config.random_seed)
    
    # 2. 架构师的性能魔法：极速生成等相关矩阵 (Equicorrelated Matrix)
    # 警告：不要用 np.random.multivariate_normal！构建 1000x1000 协方差矩阵极慢。
    # 数学戏法：X_ij = sqrt(rho) * Z_shared + sqrt(1-rho) * Z_independent
    Z_shared = rng.standard_normal((config.n_samples, 1))
    Z_independent = rng.standard_normal((config.n_samples, config.p_features))
    
    X = np.sqrt(config.rho) * Z_shared + np.sqrt(1 - config.rho) * Z_independent
    
    # 3. 生成真实的回归系数 (True Beta)
    beta_true = np.zeros(config.p_features)
    # 随机挑选 num_nonzero 个位置
    nonzero_indices = rng.choice(config.p_features, config.num_nonzero, replace=False)
    # 赋予 N(0,1) 的权重
    beta_true[nonzero_indices] = rng.standard_normal(config.num_nonzero)
    
    # 4. 计算纯净信号 (Pure Signal)
    signal = X @ beta_true
    
    # 5. 根据目标 SNR 逆向推导噪音的标准差 sigma
    # 核心数学公式：SNR = Var(Signal) / Var(Noise)  => Var(Noise) = Var(Signal) / SNR
    signal_variance = np.var(signal)
    noise_variance = signal_variance / config.snr
    sigma = np.sqrt(noise_variance)
    
    # 6. 生成高斯噪音并合成最终的 y
    noise = rng.normal(loc=0.0, scale=sigma, size=config.n_samples)
    y = signal + noise
    
    return X, y, beta_true

In [2]:
def create_experiment_suite():
    """包装调用：快速生成图片中要求的三种 SNR 环境"""
    
    print("\033[1m--- Generating Simulation Suite ---\033[0m")
    
    # 1. Low SNR (< 1, e.g., 0.5)
    config_low = SimConfig(snr=0.5, random_seed=101)
    X_low, y_low, beta_low = generate_simulation_data(config_low)
    print(f"[Low SNR]   Var(y): {np.var(y_low):.2f}, True sparsity: {np.sum(beta_low != 0)}")
    
    # 2. Medium SNR (≈ 1.0)
    config_med = SimConfig(snr=1.0, random_seed=102)
    X_med, y_med, beta_med = generate_simulation_data(config_med)
    print(f"[Med SNR]   Var(y): {np.var(y_med):.2f}, True sparsity: {np.sum(beta_med != 0)}")
    
    # 3. High SNR (> 2, e.g., 3.0)
    config_high = SimConfig(snr=3.0, random_seed=103)
    X_high, y_high, beta_high = generate_simulation_data(config_high)
    print(f"[High SNR]  Var(y): {np.var(y_high):.2f}, True sparsity: {np.sum(beta_high != 0)}")
    
    return {
        "low": (X_low, y_low, beta_low),
        "medium": (X_med, y_med, beta_med),
        "high": (X_high, y_high, beta_high)
    }

# 直接运行测试
experiment_data = create_experiment_suite()

--- Generating Simulation Suite ---
[Low SNR]   Var(y): 25.26, True sparsity: 20
[Med SNR]   Var(y): 50.45, True sparsity: 20
[High SNR]  Var(y): 14.68, True sparsity: 20


In [5]:
experiment_data["high"][2]


array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        , -0.3244679 ,  0.        ,  0.        ,
        0.        ,  0.76125338,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.15310165,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.  

In [4]:
from unilasso import *


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso as SklearnLasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import warnings

# 忽略因为极小 lambda 导致的收敛警告
warnings.filterwarnings("ignore")

def run_benchmark_and_plot(X_full, y_full, title_suffix=""):
    """
    运行 UniLasso 与 标准 Lasso 的基准对比测试，并绘制误差曲线。
    """
    print(f"\n\033[1m=== 开始基准测试: {title_suffix} ===\033[0m")
    
    # 1. 架构师的铁律：严格切分训练集与测试集 (50% / 50%)
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y_full, test_size=0.5, random_state=999
    )
    
    # ==========================================
    # 选手 A: UniLasso 出场
    # ==========================================
    # 让 UniLasso 自己去探路，生成 lambda 路径并计算所有权重
    unilasso_res = fit_unilasso(X_train, y_train, family='gaussian', lmdas=None)
    lmdas = unilasso_res.lmdas
    n_lmdas = len(lmdas)
    
    uni_train_errors = np.zeros(n_lmdas)
    uni_test_errors = np.zeros(n_lmdas)
    
    # 计算 UniLasso 在整条路径上的训练和测试误差
    for i in range(n_lmdas):
        # 假设你的 predict 函数支持传入 lmda_idx
        # 如果不支持，可以直接用: y_hat = X @ unilasso_res.coefs[i] + unilasso_res.intercept[i]
        y_train_pred = X_train @ unilasso_res.coefs[i] + unilasso_res.intercept[i]
        y_test_pred = X_test @ unilasso_res.coefs[i] + unilasso_res.intercept[i]
        
        uni_train_errors[i] = mean_squared_error(y_train, y_train_pred)
        uni_test_errors[i] = mean_squared_error(y_test, y_test_pred)

    # 找到 UniLasso 的最优测试误差及其对应的 lambda
    uni_best_idx = np.argmin(uni_test_errors)
    uni_best_test_err = uni_test_errors[uni_best_idx]
    
    # ==========================================
    # 选手 B: 标准 Lasso (scikit-learn) 出场
    # ==========================================
    std_train_errors = np.zeros(n_lmdas)
    std_test_errors = np.zeros(n_lmdas)
    
    # 强制标准 Lasso 使用完全相同的 lambda 路径 (在 sklearn 中称为 alpha)
    for i, alpha in enumerate(lmdas):
        # 架构师数学推演：sklearn的公式是 1/(2n) * ||y - Xw||^2 + alpha * ||w||_1
        # 这与你们底层 C++ 引擎的数学目标函数完全一致！可以直接将 lambda 当作 alpha 传入。
        std_lasso = SklearnLasso(alpha=alpha, fit_intercept=True, max_iter=2000)
        std_lasso.fit(X_train, y_train)
        
        y_train_pred = std_lasso.predict(X_train)
        y_test_pred = std_lasso.predict(X_test)
        
        std_train_errors[i] = mean_squared_error(y_train, y_train_pred)
        std_test_errors[i] = mean_squared_error(y_test, y_test_pred)

    # 找到 标准 Lasso 的最优测试误差
    std_best_idx = np.argmin(std_test_errors)
    std_best_test_err = std_test_errors[std_best_idx]
    
    print(f"UniLasso 最优测试误差: {uni_best_test_err:.4f} (at log-lambda: {np.log(lmdas[uni_best_idx]):.2f})")
    print(f"标准 Lasso 最优测试误差: {std_best_test_err:.4f} (at log-lambda: {np.log(lmdas[std_best_idx]):.2f})")

    # ==========================================
    # 最终呈现：全息全景可视化
    # ==========================================
    plt.figure(figsize=(12, 7))
    log_lmdas = np.log(lmdas)
    
    # 绘制 UniLasso
    plt.plot(log_lmdas, uni_train_errors, 'b--', linewidth=2, label='UniLasso - Train Error')
    plt.plot(log_lmdas, uni_test_errors, 'b-', linewidth=2, label='UniLasso - Test Error')
    # 标注最优星号
    plt.plot(log_lmdas[uni_best_idx], uni_best_test_err, 'b*', markersize=15, 
             label=f'UniLasso Best ({uni_best_test_err:.2f})')
    
    # 绘制标准 Lasso
    plt.plot(log_lmdas, std_train_errors, 'r--', linewidth=2, label='Standard Lasso - Train Error')
    plt.plot(log_lmdas, std_test_errors, 'r-', linewidth=2, label='Standard Lasso - Test Error')
    # 标注最优星号
    plt.plot(log_lmdas[std_best_idx], std_best_test_err, 'r*', markersize=15, 
             label=f'Std Lasso Best ({std_best_test_err:.2f})')
    
    plt.xlabel(r'$\log(\lambda)$ (Regularization Strength)', fontsize=14)
    plt.ylabel('Mean Squared Error (MSE)', fontsize=14)
    plt.title(f'Train vs Test Error Comparison: UniLasso vs Standard Lasso\n{title_suffix}', fontsize=16)
    
    # 翻转 X 轴 (让高惩罚在左边，低惩罚在右边，符合从左向右放松惩罚的直觉)
    plt.xlim(max(log_lmdas), min(log_lmdas))
    
    plt.legend(fontsize=12, loc='upper left')
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.show()

# ==========================================
# 调用执行区：拿我们之前写的配置表来试手！
# ==========================================
# 生成一份中等信噪比的硬核测试数据 (样本量放大到 600，保证切分后各有 300)
config = SimConfig(n_samples=600, p_features=1000, rho=0.5, num_nonzero=20, snr=1.0)
X_sim, y_sim, _ = generate_simulation_data(config)

# 跑起来！
run_benchmark_and_plot(X_sim, y_sim, title_suffix="Medium SNR (SNR=1.0, p=1000)")


=== 开始基准测试: Medium SNR (SNR=1.0, p=1000) ===


 91%|          | 0/100 [00:00:00<?, ?it/s]█████████ | 91/100 [00:00:00<00:00:00, 1719.96it/s] [dev:90.1%]


ValueError: No regularization strengths remain after removing invalid values